# DengueWatch BD: your voice + your face → the launch videos

This notebook turns **your own voice and face** into the three launch videos (citizen tutorial, city corporation invitation, your story). Run the cells top to bottom.

| Step | What happens | Tool |
| --- | --- | --- |
| 1 | You upload 6 short voice clips reading fixed sentences (one per emotion) | |
| 2 | Every narration line in `media/scripts/*.json` is spoken **in your voice**, in the emotion of that scene | AI4Bharat **IndicF5** (MIT) |
| 3 | Captions are aligned word by word to the new audio (text = the script, so no recognition errors) | Meta **MMS** forced aligner |
| 4 | You upload a 30–60 s video of yourself; your lips are re-synced to each story line | ByteDance **LatentSync** (Apache-2.0) |
| 5 | The videos are rendered with the motion graphics and real app screens | **Remotion** |
| 6 | You download the MP4s and `.srt` subtitle files | |

Nothing is "trained": IndicF5 and LatentSync copy your voice and mouth movement from the clips at generation time. Your clips stay in this Colab session and are deleted when it ends. Nothing is uploaded to GitHub.

**Runtime → Change runtime type → GPU.** The free T4 works (LatentSync 1.5, softer mouth). An L4 or A100 (Colab Pro) uses LatentSync 1.6, which is sharper. Expect about 20 min for voices and about 30–60 min for lip-sync.

Recording instructions: [`media/RECORDING_GUIDE.md`](https://github.com/scornik/Dengue-Watch-BD/blob/claude/affectionate-feynman-lr76e4/media/RECORDING_GUIDE.md).

In [ ]:
#@title 0 · Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: Runtime → Change runtime type → GPU"

In [ ]:
#@title 1 · Settings and code
BRANCH = "claude/affectionate-feynman-lr76e4"  #@param {type:"string"}
# ↑ the branch with the video tools; switch to "main" once it is merged
SITE_URL = "dengue-watch-bd.vercel.app"  #@param {type:"string"}
YOUR_NAME = ""  #@param {type:"string"}
CONTACT = ""  #@param {type:"string"}
VIDEOS = "story,citizen,govt"  #@param {type:"string"}

import os, subprocess, json, glob, shutil
os.chdir("/content")
if not os.path.exists("Dengue-Watch-BD"):
    !git clone -q --depth 1 -b {BRANCH} https://github.com/scornik/Dengue-Watch-BD
os.chdir("/content/Dengue-Watch-BD")
!pip -q install uv
!git log --oneline -1

## 2 · Your voice

Record **each sentence below** on your phone (voice recorder app), in a quiet room, phone about 20 cm from your mouth. Read it **in the emotion named**, as you would say it to a friend. Each clip should be 6–12 seconds. Name the files `serious.m4a`, `warm.m4a`, … (any audio format works).

The emotion of every narration line comes from the matching clip. A tired, flat clip gives a tired, flat video.

In [ ]:
#@title 2a · The sentences to record (read exactly these words)
REF = {
  "serious":   "আমাদের চারপাশে জমে থাকা পানিতেই জন্মায় এডিস মশা। প্রতিদিন অনেক মানুষ ডেঙ্গুতে আক্রান্ত হচ্ছেন, হাসপাতালে জায়গা নেই।",
  "warm":      "কাজটা খুব সহজ। ফোনে একটা ছবি তুলুন, মানচিত্রে জায়গাটা দেখিয়ে দিন, তারপর পাঠিয়ে দিন। এক মিনিটও লাগে না।",
  "calm":      "প্রতিটি রিপোর্টে ছবি, অবস্থান আর পাত্রের ধরন থাকে। সব তথ্য সবার জন্য খোলা, যে কেউ দেখে নিতে পারেন।",
  "confident": "এই প্ল্যাটফর্ম ওপেন সোর্স। প্রতিটি ওয়ার্ডের অগ্রগতি সবাই দেখতে পান, আর তথ্য বিনা মূল্যে ডাউনলোড করা যায়।",
  "hopeful":   "আমরা সবাই মিলে চেষ্টা করলে এই মশাকে থামানো সম্ভব। আজই আপনার বাড়ির আশেপাশে একবার খুঁজে দেখুন।",
  "energetic": "চলুন শিকারে নামি! কাছের জায়গাটা খুঁজে নিন, পানি ফেলে দিন, ছবি তুলুন আর পয়েন্ট জিতে নিন!",
}
for mood, text in REF.items():
    print(f"{mood:10}  {text}")

In [ ]:
#@title 2b · Upload the clips (serious.*, warm.*, calm.*, confident.*, hopeful.*, energetic.*)
from google.colab import files
from IPython.display import Audio, display
REFS = "media/voice/refs"
os.makedirs(REFS, exist_ok=True)
up = files.upload()
for name in up:
    mood = os.path.splitext(name)[0].lower()
    if mood not in REF:
        print("skipped (name it after a mood):", name); continue
    out = f"{REFS}/{mood}.wav"
    # mono 24 kHz, trim silence at both ends, gentle loudness normalisation
    !ffmpeg -y -loglevel error -i "{name}" -ac 1 -ar 24000 -af "silenceremove=start_periods=1:start_threshold=-45dB,areverse,silenceremove=start_periods=1:start_threshold=-45dB,areverse,loudnorm" "{out}"
    open(f"{REFS}/{mood}.txt", "w").write(REF[mood])
    os.remove(name)
if os.path.exists(f"{REFS}/warm.wav"):
    shutil.copy(f"{REFS}/warm.wav", f"{REFS}/default.wav"); shutil.copy(f"{REFS}/warm.txt", f"{REFS}/default.txt")
for w in sorted(glob.glob(f"{REFS}/*.wav")):
    print(w); display(Audio(w))
missing = [m for m in REF if not os.path.exists(f"{REFS}/{m}.wav")]
print("Missing moods (will use warm):", missing or "none")

## 3 · Hugging Face access (one time)

IndicF5 is free, but you must accept its terms once. Access is granted **instantly**.

1. Sign in at [huggingface.co](https://huggingface.co) (or create a free account).
2. Open **[huggingface.co/ai4bharat/IndicF5](https://huggingface.co/ai4bharat/IndicF5)** and click **"Agree and access repository"**. The page should then show the model files.
3. Create a token at **[Settings → Access tokens](https://huggingface.co/settings/tokens)** → *Create new token* → type **Read** → copy it.
   (A *fine-grained* token also works only if you tick "Read access to contents of all public gated repos you can access".)
4. Run the cell below and paste the token. Use the **same account** you clicked "Agree" with.

In [ ]:
#@title 3 · Log in and check IndicF5 access
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
#@title 3b · Check access (run after logging in)
from huggingface_hub import hf_hub_download, whoami
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError
try:
    print("Logged in as:", whoami()["name"])
    hf_hub_download("ai4bharat/IndicF5", "config.json")
    print("✅ IndicF5 access OK. Continue to step 4.")
except GatedRepoError:
    print("❌ This account has not accepted the IndicF5 terms yet.")
    print("   Open https://huggingface.co/ai4bharat/IndicF5 while signed in as the account above,")
    print("   click 'Agree and access repository', then run this cell again.")
except HfHubHTTPError as e:
    print("❌", e)
    print("   If you used a fine-grained token, create a classic 'Read' token instead and run step 3 again.")
except Exception as e:
    print("❌ Not logged in:", e, "
   Run step 3 and paste your token.")

In [ ]:
#@title 4 · Install the voice tools (≈5 min)
!cd media/voice && uv venv -q -p 3.10 .venv && uv pip install -q --python .venv/bin/python -r requirements-clone.txt --extra-index-url https://download.pytorch.org/whl/cu121
!media/voice/.venv/bin/python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
#@title 5 · Speak every line in your voice + align captions
for v in VIDEOS.split(","):
    !media/voice/.venv/bin/python media/voice/narrate.py {v.strip()} --engine indicf5

In [ ]:
#@title 5b · Listen
VIDEO = "story"  #@param ["story", "citizen", "govt"]
m = json.load(open(f"media/video/src/generated/{VIDEO}.json"))
for s in m["scenes"]:
    print(s["id"], "·", s["mood"], "·", " ".join(w["text"] for w in s["words"]))
    display(Audio(f"media/video/public/{s['audio']}"))

In [ ]:
#@title 5c · Redo one line (generation is random: run again until it sounds like you)
VIDEO = "story"  #@param ["story", "citizen", "govt"]
SCENE = "wife"  #@param {type:"string"}
!media/voice/.venv/bin/python media/voice/narrate.py {VIDEO} --engine indicf5 --only {SCENE}
s = next(x for x in json.load(open(f"media/video/src/generated/{VIDEO}.json"))["scenes"] if x["id"] == SCENE)
display(Audio(f"media/video/public/{s['audio']}"))

## 6 · Your face (optional but strongest for the story)

Record **30–60 seconds** of yourself talking to the camera about anything, in the same place and light you'd want in the video:
- phone at eye level on a stand, landscape or portrait, 1080p, 30 fps
- face fully visible, no hand in front of the mouth, no mask or sunglasses
- natural movement: nod, blink, look at the lens; don't read from a screen beside the camera

LatentSync keeps your real head and eyes and only re-draws the mouth for each line.

In [ ]:
#@title 6 · Upload your face video and lip-sync
LIPSYNC_VIDEOS = "story"  #@param {type:"string"}
CITIZEN_SCENES = "hook cta"  #@param {type:"string"}
up = files.upload()
face = list(up)[0]
os.makedirs("media/avatar", exist_ok=True)
shutil.move(face, "media/avatar/me" + os.path.splitext(face)[1])
face = glob.glob("media/avatar/me.*")[0]
vram = int(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"]).split()[0])
os.environ["LATENTSYNC_VERSION"] = "1.6" if vram >= 20000 else "1.5"
print("LatentSync", os.environ["LATENTSYNC_VERSION"], "for", vram, "MiB VRAM")
for v in LIPSYNC_VIDEOS.split(","):
    v = v.strip()
    scenes = CITIZEN_SCENES if v == "citizen" else ""
    !bash media/avatar/lipsync.sh {v} {face} {scenes}

In [ ]:
#@title 7 · Render the videos (≈10–20 min each on Colab CPUs)
NODE = "/content/node/bin"
if not os.path.exists(NODE):
    !curl -sL https://nodejs.org/dist/v22.11.0/node-v22.11.0-linux-x64.tar.xz | tar xJ -C /content && mv /content/node-v22.11.0-linux-x64 /content/node
    !apt-get -qq install -y libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libgbm1 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libasound2 libpango-1.0-0 libcairo2 > /dev/null
os.environ["PATH"] = NODE + ":" + os.environ["PATH"]
!cd media/video && npm ci --no-audit --no-fund --loglevel=error
props = {"siteUrl": SITE_URL}
ids = {"citizen": "Citizen", "govt": "Govt", "story": "Story"}
for v in VIDEOS.split(","):
    v = v.strip()
    p = dict(props)
    if v == "govt": p["contact"] = CONTACT
    if v == "story": p.update({"name": YOUR_NAME, "role": "ডেঙ্গুওয়াচ বিডির নির্মাতা"})
    pj = json.dumps(p, ensure_ascii=False)
    !cd media/video && npx remotion render {ids[v]} out/{v}.mp4 --codec=h264 --crf=18 --props='{pj}'

In [ ]:
#@title 8 · Download (videos + subtitle files)
!cd media/video/out && zip -q -r /content/denguewatch-videos.zip *.mp4 *.srt && ls -lh /content/denguewatch-videos.zip
files.download("/content/denguewatch-videos.zip")

### Before you post
- Watch every video once with sound and once muted (captions only).
- Add to the post text: **"ভয়েস ও ঠোঁট মেলানো এআই দিয়ে তৈরি"** (voice and lip-sync made with AI). Meta requires a label on realistic AI-made video.
- Credit the demo photos: OakleyOriginals, chris.rycroft (CC BY 2.0).
- Upload the `.srt` as Bengali subtitles on Facebook/YouTube (Facebook wants a name like `citizen.bn_BD.srt`).